In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import pathlib
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

E0000 00:00:1750192707.865223   72370 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750192707.870094   72370 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750192707.883097   72370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750192707.883128   72370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750192707.883130   72370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750192707.883132   72370 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [3]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Available Devices :  [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]


I0000 00:00:1750192715.101683   72370 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9706 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


# Chapter 11: Deep Learning Text

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we look at the particular types of deep network architectures that work well when processing textual time series, as
well as other aspects specific to preparing and processing textual input.

Reload datasets and defined constants from previous notebook(s) used here.

In [4]:
base_dir = pathlib.Path("../data/aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    train_dir, batch_size=batch_size
)

val_ds = keras.utils.text_dataset_from_directory(
    val_dir, batch_size=batch_size
)

test_ds = keras.utils.text_dataset_from_directory(
    test_dir, batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [5]:
# we will create sequences, but will make the maximum sequence length of 600 tokens.
# this means longer reviews will get choped to first 600 words, and shorts ones will be
# filled with the mask index 0 token
max_length = 600

# but we will still use most frequent 20,000 words for the vocabulary
max_tokens = 20000

# we'll truncate inputs after firt 600 words, this is a reasonable
# choice since the average review is 233 words and only 5%
# of reviews are longer than 600
# also since we are using sequences, bigram doesn't make sense, so we
# go back to 1-gram vocabulary
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)

# prepare a dataset that only yields raw text inputs (no labels)
text_only_train_ds = train_ds.map(lambda x, y: x)

# build the vocabulary again
text_vectorization.adapt(text_only_train_ds)

# The dataset instances again on the sequence output
# if we looked, what output would you expect from these,
# should be (32, 600) shaped outputs for batch size 32
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

## 11.4 The Transformer Architecture

- Starting 2017 a new model architecture started overtaking recurrent neural networks across
  most NLP tasks: The Transformer
- Seminal paper: Ashish Vaswani et al., “Attention is all you need” (2017), https://arxiv.org/abs/1706.03762
- A simple mechanism called **neural attention** could be used to build powerful sequence models
  that didn't feature any recurrent layers or convolution layers.
- Neural attention has fast become one of the most influential ides in deep learning.

### 11.4.1 Understanding Self-attention

Pseudo-code implementation of self-attention

```python
def self_attention(input_sequence):
    """The input_sequence is a sequence of encoded vectors from
    a word embedding like space.  For example, for 600 words encoded
    in a 100d embedding, it would be shape (600, 100)

    Parameters
    ----------
    input_sequence : numpy array shape (max_tokens, dimensions)
        A single sequence, like a sentence for text, but encoded in a word embedding like space. Each
        sample is the vector of embeddings for a single token.

    Returns
    -------
    output : numpy array shape (max_tokens, dimensions)
        Same shape array of token vectors returned, but all of the token vectors has been weighted and
        summed by attention, so each is now a context-aware token vector.
    """
    output = np.zeros(shape=input_sequence.shape)
    # iterate over each individual token from 0..max_tokens-1
    for i, pivot_vector in enumerate(input_sequence):

        # step 1 compute relevancy scores e.g. attention 
        # compute the attention score, which is dot product between the token
        # and every other token in this input sequence
        scores = np.zeros(shape=(len(input_sequence),))
        for j, vector in enumerate(input_sequence):
            scores[j] = np.dot(pivot_vector, vector.T)

        # scale by a normalization factor and apply softmax, result is probability
        # distribution that sums up to 1, but still basically importance scores
        scores /= np.sqrt(input_sequence.shape[1])
        scores = softmax(scores)

        # now step 2, compute sum of all word vectors, weighted
        # by relevancy, resulting in new representation of this token
        new_pivot_representation = np.zeros(shape=pivot_vector.shape)
        for j, vector in enumerate(input_sequence):
            new_pivot_representation += vector * scores[j]

        # so on output, the vector representation for token i is the new 
        # context-aware representation
        output[i] = new_pivot_representation
    return output
```

Keras has a built-in layer to handle learning and using a self-attention embedding:

In [7]:
num_heads = 4
embed_dim = 100

inputs = keras.Input(shape=(64,), dtype="int64")
mha_layer = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
outputs = mha_layer(inputs, inputs, inputs)

If you look closely you might notice:

- Why are we passing the inputs to the layer *three* times?
- What are these "multiple heads" we're referring to?

#### Generalized self-attention: the query-key-value model

### 11.4.2 Multi-head Attention

### 11.4.3 The Transformer encoder

- If adding extra dense projections (in multi-heads) is so useful, why not to the output of the attention mechanism?
- Might want to also add residual connections since model is starting to do a lot
- Normalization layers are also supposed to help gradients flow better during backpropagation.

Roughly the architecture of Transformer encoder

- Factoring outputs into multiple independent spaces
- Adding residual connections
- Adding normalization layers

Together these form the **Transformer encoder**  one of two critical parts that make up the transformer architecture.

Let's implement a Transformer encoder and try it on the movie review sentiment classification.

Here our example form the text shows implementing a `TransformerEncoder` by hand, using subclassing from `keras.layers.Layer` as
we have seen examples of previously. 

In [8]:
class TransformerEncoder(layers.Layer):
    """Example implementation of full TransformerEncoder architecture by subclassing
    the keras.layers.Layer class.  
    """
    
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        """Override base constructor to support TransformerEncoder initialization parameters

        Parameters
        ----------
        embed_dim : int
            The size of the embedding dimension, e.g. the size of the input sample token vectors
        dense_dim : int
            The size of the inner dense layer for the Dense projectios of the output of the MultiHead
        num_heads : int
            The number of independent heads (features) in the MultiHeadAttention to use
        """
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads

        # create the multihead attention layer use built-in keras layer
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        # Use a single Dense dimension for encoding,
        # final Dense layer projects back to output embeding dimension
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )

        # the layer normaliztion layers, we will put one between the multi-head and dense projections
        # then one after dense proejcts before output from this layer
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        """Computation goes in call member function.

        Parameters
        ----------
        inputs : ndarray (num_samples, max_tokens, embedding_dim)
            The inputs for Keras layers always have some number of samples.  In our example
            we use token sequences where each token is embeded in an embedding space of embedding_dim
            number of dimensions.
        mask : ndarray (num_samples, max_tokens, embedding_dim)?
            The attention_mask goes in as one of the 3 inputs to the multihead attention layer.
        """
        # the mask generated by the Emebdding layer will be 2D but the attention layer
        # expects to be 3D or 4D so we expand its rank
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        # The attention layer is the MultiHeadAttention
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        # the output from attention is fed into first normalization layer,
        # notice residual connection of the original inputs is also added in here
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        # likewise residual connection around the dense layers is added back in here
        return self.layernorm_2(proj_input + proj_output)
    
    def get_config(self):
        """Implement serialization so that we can save the model. Should return a
        python dict that contains values of the constructor arguments used to create the
        layer.  So we chain the super class get_config, then add in our 3 additional
        constructor arguments.
        """
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

You'll note we are using `LayerNormalization` instead of `BatchNormalization` between the layers in our
subclassed layer.  This is because `BatchNormalization` does not work well for sequence.  Instead
`LayerNormalization` normalizes each sequence independently from other sequences in the batch.

In NumPy like psuedocode, `LayerNormalizaiton` does something like:

In [9]:
def layer_normalization(batch_of_sequences):
    """Independent normalization of sequences

    Parameters
    ----------
    batch_of_sequences : ndarray (batch_size, sequence_length, embedding_dim)
        The typical batch for a sequence using an embedding space.
    """
    # the axis=-1 means we only pool data over the last axis, thus we get
    # a shape here of (batch_size, sequence_length, 1) of the means and variance
    mean = np.mean(batch_of_sequences, keepdims=True, axis=-1)
    variance = np.var(batch_of_sequences, keepdims=True, axis=-1)
    print(mean.shape, variance.shape)
    # result is each individual token vector is normalized
    return (batch_of_sequences - mean) / variance

# random data of a batch of 64 items, sequence lengths 600 and embedding dimension 100
batch_of_sequences = np.random.randn(64, 600, 100)
n = layer_normalization(batch_of_sequences)
n.shape


(64, 600, 1) (64, 600, 1)


(64, 600, 100)

As a reminder, a `BatchNormalization` layer does something like the following when normalizing over all batches:

In [10]:
def batch_normalization(batch_of_images):
    """Normalization of batches

    Parameters
    ----------
    batch_of_images : ndarray (batch_size, height, width, channels)
    """
    # pool over the batch axis (axis 0) which creates interactions between samples in batches
    # thus we get a shape her of (1, 1, 1, 3) of the means and variances
    mean = np.mean(batch_of_images, keepdims=True, axis=(0, 1, 2))
    variance = np.var(batch_of_images, keepdims=True, axis=(0, 1, 2))
    print(mean.shape, variance.shape)
    # result is all samples are normalized for each channel of input
    return (batch_of_images - mean) / variance

# random data of a batch of 64 items
batch_of_images = np.random.randn(64, 1024, 768, 3)
n = batch_normalization(batch_of_images)
n.shape

(1, 1, 1, 3) (1, 1, 1, 3)


(64, 1024, 768, 3)

While BatchNormalization collects information from many samples to obtain accurate
statistics for the feature means and variances, LayerNormalization pools data
within each sequence separately, which is more appropriate for sequence data.

No that we have our custom `TransformerEncoder` layer, we can use it to assemble
a text-classification model similar to the LSTM based models we used in previous notebook.

In [11]:
vocab_size = 20000
embed_dim = 256
num_heads = 2
dense_dim = 32

# model, with embedding layer to learn a 256 dimension embedding,
# followed by the custom TransformerEncoding layer discussed
inputs = keras.Input(shape=(None,), dtype="int64")
x = layers.Embedding(vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, None, 256)      │       543,776 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,664,033 (21.61 MB)

 Trainable params: 5,664,033 (21.61 MB)

 Non-trainable params: 0 (0.00 B)

Let's train it as before.

In [12]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-transformer-encoder-custom.keras", save_best_only=True)
]

model.fit(int_train_ds, 
          validation_data=int_val_ds,
          epochs=20, 
          callbacks=callbacks)

Epoch 1/20


I0000 00:00:1750198264.503256   72494 service.cc:152] XLA service 0x76e05005a620 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750198264.503309   72494 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
I0000 00:00:1750198265.053043   72494 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1750198285.076535   72494 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 80s 91ms/step - accuracy: 0.5853 - loss: 0.8171 - val_accuracy: 0.8254 - val_loss: 0.3911
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8242 - loss: 0.3924 - val_accuracy: 0.8426 - val_loss: 0.3527
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8490 - loss: 0.3434 - val_accuracy: 0.8510 - val_loss: 0.3437
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - accuracy: 0.8672 - loss: 0.3105 - val_accuracy: 0.8584 - val_loss: 0.3320
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8763 - loss: 0.2928 - val_accuracy: 0.8604 - val_loss: 0.3291
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8862 - loss: 0.2703 - val_accuracy: 0.8662 - val_loss: 0.3225
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.8970 - loss: 0.2550 - val_accuracy: 0.8624 - val_loss: 0.3234
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.9090 - loss: 0.2335 - val_accurac

In [13]:
# note that have to give extra information for load_model when using custom layers,
# this is where the code in the get_config() method is needed
model = keras.models.load_model(
    "../models/ch11-transformer-encoder-custom.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder})

# NOTE: getting warning build() was called on layer transformer_encoder, however the layer does ot have a build() method
# Keras API may have changed, maybe if we just add in the missing build() method, though need to check if really the stuff in
# the constructor should instead be in the build?
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

/opt/conda/lib/python3.12/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'transformer_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.8645 - loss: 0.3211
Test acc: 0.865


You will find the transformer architecture gets only 87% accuracy or so again, still below bag-of-word models.

Lets do the same model, but use the now available KerasHub `TransformerEncoder` instead of our custom layer implementation.  The documentation
for the [KerasHub TransformerEncoder](https://keras.io/keras_hub/api/modeling_layers/transformer_encoder/) if you need to read it.

**Note**: I need to do some more digging, but the KerasHub library seems to be newer stuff that hasn't made it into the base Keras.  So
you need to do a

```
$ python3 -m pip install keras-hub
```

to get this set of additional libraries, and then do the

```
import keras_hub
```

for the following example.

In [14]:
import keras_hub
vocab_size = 20000
embed_dim = 256
num_heads = 2
dense_dim = 32

# model, with embedding layer to learn a 256 dimension embedding,
# followed by the custom TransformerEncoding layer discussed
inputs = keras.Input(shape=(None,), dtype="int64")
x = layers.Embedding(vocab_size, embed_dim)(inputs)
# the built-in layer only requires the intermediate_dim (what we call the number of dimensions for our dense
# layer), and the num_heads.  The embed_dim is probably inferred from inputs.
#x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = keras_hub.layers.TransformerEncoder(dense_dim, num_heads)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_1           │ (None, None, 256)      │       280,864 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,401,121 (20.60 MB)

 Trainable params: 5,401,121 (20.60 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-transformer-encoder-hub.keras", save_best_only=True)
]

model.fit(int_train_ds, 
          validation_data=int_val_ds,
          epochs=20, 
          callbacks=callbacks)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 78s 95ms/step - accuracy: 0.5675 - loss: 0.8644 - val_accuracy: 0.8244 - val_loss: 0.3902
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 85ms/step - accuracy: 0.8266 - loss: 0.3931 - val_accuracy: 0.8410 - val_loss: 0.3601
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 85ms/step - accuracy: 0.8485 - loss: 0.3476 - val_accuracy: 0.8544 - val_loss: 0.3393
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 84ms/step - accuracy: 0.8633 - loss: 0.3201 - val_accuracy: 0.8404 - val_loss: 0.3615
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 84ms/step - accuracy: 0.8719 - loss: 0.3010 - val_accuracy: 0.8574 - val_loss: 0.3296
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 84ms/step - accuracy: 0.8867 - loss: 0.2740 - val_accuracy: 0.8606 - val_loss: 0.3214
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 84ms/step - accuracy: 0.8989 - loss: 0.2499 - val_accuracy: 0.8614 - val_loss: 0.3272
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 83ms/step - accuracy: 0.9066 - loss: 0.2326 - 

In [16]:
model = keras.models.load_model("../models/ch11-transformer-encoder-hub.keras")

print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.8660 - loss: 0.3097
Test acc: 0.868


At this point you may feel a bit uneasy about the results, maybe even a bit cheated.  Isn't the conclusion supposed to be that the sequence
models, including the new shiny Transformer encoder architecture, should perform better than the bag-of-word models that ignore word order?

We started off the discussion of sequence models by highlighting the importnace of word order.  The text said that the Transformer
was a sequence-processing architecture, originally developed for machine translation.  And yet, the Transformer encoder we just created
wasn't a sequence model at all! Did you notice?  The MultiHead and extra layers we added in the Transformer architecture have none of
the (conceptually) iterative part like the RNN to process each token in sequence.  The Transformer we custom built (and the newly added
bult-in lyaer) is composed of the Multi-head attention module, that looks at the tokens in a sequence as a set, and some additional dense and normalizing layers
that all process sequence of tokens independently and in batches.  You could change the order of the tokens in a sequence and you'd get
the exact same pairwise attention scores and the exact same context-aware representations.

If you were to completely scramble the words in every movie review, the model wouldn’t notice,
and you’d still get the exact same accuracy. Self-attention is a set-processing mechanism,
focused on the relationships between pairs of sequence elements.

So why do we say that Transformer is a sequence
model? And how could it possibly be good for machine translation if it doesn’t look
at word order? 

Transformer was a hybrid approach that is technically order-agnostic, but that manually
injects order information in the representations it processes. 
Have to add **positional encoding**.

#### Using positional encoding to re-inject order information

Give the model access to word order information, add the word's position in the sentence to each word embedding.

Input word embeddings will have two components: the usual word vector, which represents the word independently of any specific context, and a position
vector, which represents the position of the word in the current sentence.

Not ideal to simply integer position of token, because positions can be potentially very large integers which will disrupt the range of values in the embeddings.

We’ll learn position embedding
vectors the same way we learn to embed word indices. We’ll then proceed
to add our position embeddings to the corresponding word embeddings, to obtain a
position-aware word embedding. This technique is called “positional embedding.”

The following is another `Layer` subclass example to implement this technique.  As with the `TransformerEncoder` the `PositionEmbedding` layer has been
added in to more recent versions of the Keras API.  We will first show creating by hand, then use the built-in version.

In [17]:
class MyTfNotEqualLayer(layers.Layer):
    def call(self, inputs):
        return tf.math.not_equal(inputs, 0)

class PositionalEmbedding(layers.Layer):
    """Example implementation of learning position embedding vectors as a layer.
    """
    
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        """Override base class constructor to add in additional parameters
        needed for this layer.

        Parameters
        ----------
        sequence_length : int
            A downside of position embeddings is that the sequence length needs to be known in advance.
        input_dim : int
        output_dim : int
            The number of input dimensions coming into this embedding layer, and the number of output dimensions to
            create and output.
        """
        # chain superclass constructor first
        super().__init__(**kwargs)

        # prepare an embedding layer for the token indices
        self.token_embeddings = layers.Embedding(input_dim=input_dim, output_dim=output_dim)

        # add another embedding layer for the token positions
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=output_dim)
        
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim
    
    def call(self, inputs):
        """Computation is handled by call member method

        Parameters
        ----------
        inputs : ndarray (num_samples, sequence_length, input_dim)
            The inputs for Keras layers always have some number of samples.  In our example
            we use token sequences of sequence_length where each token is embeded in an
            embedding space of input_dim number of dimensions.        
        """
        # positions is a sequence from 0,1,...length-1 of the inputs
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)

        # perform embedding learning for the tokens from the input
        embedded_tokens = self.token_embeddings(inputs)

        # perform embedding learning of positions
        embedded_positions = self.position_embeddings(positions)

        # add both embedding vectors of token and position embeddings together to be returned
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        """like embedding layer, this layer should be able to generate a mask so we can ignore
        padding 0s in the inputs.  The compute_mask method will be called automatically by the
        framework and the mask will get propagated to the next layer

        Parameters
        ----------
        inputs : ndarray (num_samples, sequence_length, input_dim)
            The inputs to generate embedding masks for.       
        """
        # a little kludgy, API has changed here apparently, we will have to return a laye that wraps this
        # tensorflow function
        return MyTfNotEqualLayer()(inputs)

    def get_config(self):
        """Implement serialization so that we can save the model. Should return a
        python dict that contains values of the constructor arguments used to create the
        layer.  So we chain the super class get_config, then add in our 3 additional
        constructor arguments.
        """
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config

You would use this self-created `PositionEmbedding` layer just like a regular `Embedding` layer.  Let's see it in action!

#### Putting it all together: A text-classification transformer

All we have to do from the previous transformer model to take word order into account is swap the old `Embedding` layer with our
position-aware version.

In [18]:
vocab_size = 20000
sequence_length = 600
embed_dim = 256
num_heads = 2
dense_dim = 32

inputs = keras.Input(shape=(None,), dtype="int64")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# NOTE: warning about TransformerEncoder was passed an input mask attached, however this layer does not support masking.
# may need to fix the above example to correctly use the mask, API might have changed.
model.summary()

/opt/conda/lib/python3.12/site-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'global_max_pooling1d_2' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  5,273,600 │ input_layer_6[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ my_tf_not_equal_la… │ (None, None)      │          0 │ input_layer_6[0]… │
│ (MyTfNotEqualLayer) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │    543,776 │ positional_embed… │
│ (TransformerEncode… │                   │            │ my_tf_not_equal_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ transformer_enco… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 256)       │          0 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │        257 │ dropout_9[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,817,633 (22.19 MB)

 Trainable params: 5,817,633 (22.19 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-full-transformer-encoder-custom.keras", save_best_only=True)
]

model.fit(int_train_ds,
          validation_data=int_val_ds,
          epochs=20,
          callbacks=callbacks)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 62ms/step - accuracy: 0.6397 - loss: 0.7464 - val_accuracy: 0.8454 - val_loss: 0.3537
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 57ms/step - accuracy: 0.8426 - loss: 0.3535 - val_accuracy: 0.7852 - val_loss: 0.5533
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 59ms/step - accuracy: 0.8837 - loss: 0.2812 - val_accuracy: 0.8594 - val_loss: 0.3335
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 59ms/step - accuracy: 0.9072 - loss: 0.2336 - val_accuracy: 0.8802 - val_loss: 0.2973
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.9234 - loss: 0.1927 - val_accuracy: 0.8834 - val_loss: 0.3923
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 58ms/step - accuracy: 0.9357 - loss: 0.1677 - val_accuracy: 0.8810 - val_loss: 0.3824
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.9494 - loss: 0.1352 - val_accuracy: 0.8818 - val_loss: 0.3980
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 57ms/step - accuracy: 0.9638 - loss: 0.1024 - 

In [20]:
model = keras.models.load_model(
    "../models/ch11-full-transformer-encoder-custom.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder, "PositionalEmbedding": PositionalEmbedding, "MyTfNotEqualLayer": MyTfNotEqualLayer})

print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

/opt/conda/lib/python3.12/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'transformer_encoder_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.8784 - loss: 0.2963
Test acc: 0.878


You should usually get an 88% test accuracy this time, a bit of an improvement over the Transformer without positional embedding that demonstrates
the value of word orderinformation for text classification.

If you are keeping track, this should usually be the best sequence model so far.  However it's still one notch below the best
bag-of-words approach.

As mentioned before, there are versions of the `TransformerEncoder` and `PositionEmbedding` layers (name was slightly changed from text) in
the KerasHub library.  Here is the same previous example using these built-in layers.

Although, if I'm reading the [KerasHub PositionEmbedding](https://keras.io/keras_hub/api/modeling_layers/position_embedding/) documentation correctly, this class only does
positional embedding, we need to have a separate embedding layer before it for the token embedding.  Our custom class
example was running the two embedding layers in parallel.
Also note we have to combine the outputs from the embeddings by hand in this network graph.

In [21]:
vocab_size = 20000
sequence_length = 600
embed_dim = 256
num_heads = 2
dense_dim = 32

inputs = keras.Input(shape=(None,), dtype="int64")
#x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
# This class assumes that in the input tensor, the last dimension corresponds to the features (we called embed_dim), 
# and the dimension before the last corresponds to the sequence (sequence_length).
token_embeddings = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
position_embeddings = keras_hub.layers.PositionEmbedding(sequence_length)(token_embeddings)
# notice from documentation we have to combine the embeddings from the previous layers
# by hand in our model
outputs = token_embeddings + position_embeddings
#x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = keras_hub.layers.TransformerEncoder(dense_dim, num_heads)(outputs)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# NOTE: warning about TransformerEncoder was passed an input mask attached, however this layer does not support masking.
# may need to fix the above example to correctly use the mask, API might have changed.
model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 256) │  5,120,000 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, None, 256) │    153,600 │ embedding_6[0][0] │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 256) │          0 │ embedding_6[0][0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │    280,864 │ add[0][0]         │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ transformer_enco… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 256)       │          0 │ global_max_pooli… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 1)         │        257 │ dropout_12[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,554,721 (21.19 MB)

 Trainable params: 5,554,721 (21.19 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-full-transformer-encoder-hub.keras", save_best_only=True)
]

model.fit(int_train_ds,
          validation_data=int_val_ds,
          epochs=20,
          callbacks=callbacks)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 66s 82ms/step - accuracy: 0.5862 - loss: 0.8151 - val_accuracy: 0.7700 - val_loss: 0.4720
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 48s 76ms/step - accuracy: 0.7897 - loss: 0.4506 - val_accuracy: 0.7874 - val_loss: 0.4490
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 76ms/step - accuracy: 0.8222 - loss: 0.3914 - val_accuracy: 0.8240 - val_loss: 0.3948
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 76ms/step - accuracy: 0.8413 - loss: 0.3522 - val_accuracy: 0.8288 - val_loss: 0.3918
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 76ms/step - accuracy: 0.8655 - loss: 0.3123 - val_accuracy: 0.8374 - val_loss: 0.3758
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.8873 - loss: 0.2708 - val_accuracy: 0.8264 - val_loss: 0.4105
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.9079 - loss: 0.2340 - val_accuracy: 0.8420 - val_loss: 0.3730
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 44s 70ms/step - accuracy: 0.9273 - loss: 0.1854 - 

In [23]:
model = keras.models.load_model(
    "../models/ch11-full-transformer-encoder-hub.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder, "PositionalEmbedding": PositionalEmbedding, "MyTfNotEqualLayer": MyTfNotEqualLayer})

print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.8410 - loss: 0.3679
Test acc: 0.843


**TODO**: need to check that, looks like not getting expected performance for the transformer
with positional layers from KerasHub layers.

### 11.4.4 When to use Sequence Models over Bag-of-words Models

- May hear that bag-of-word approach is outdated and should be using sequence models for text processing.
- This is not true, a small stack of `Dense` layers with bigrams was best we got for IMDB sentiment classifier.
- Advice from text author, calculate ratio

$$(num\_samples) / (mean(sample\_length))$$

- If ratio > 1500, then go with sequence model.
- If ratio < 1500 go with bag-of-bigrams

The argument why is:
- The input of a sequence model represents a richer and more complex space, and thus it takes more data to map out that space
- Meanwhile, a plain set of terms is a space so simple that you can train a logistic regression
  on top using just a few hundreds or thousands of samples.
- In addition, the shorter a sample is, the less the model can afford to discard any of the information it contains,
  in particular word order becomes more important.

## Summary

<font color='blue'>
    
- **Neural attention** can be used to build powerful sequence models that don't feature recurrent layers.
- Make features **context-specific** by computing attention.
- A smart embedding space provides different vector representations for a word depending on the other words around it.
- `MultiHeadAttention` layer takes 3 inputs, all using a word embedding encoding.  It outputs embeddings after factoring in self-attention for the sequences.
- `TransformerEncoder` layer uses multi-head attention as input, and factors outputs into multiple inddpendent spaces, adds residual connections and normalization layers.
- Bag-of-word models with bi-grams are sill good choices when you may not have enough samples or sample length is large.
- Sequence models, including transformers, work best with large data and/or small sample size.